# Medical Diagnosis Assistant using RAG + LangChain

**Capstone Project — Healthcare Vertical**

This notebook implements a two-retriever RAG system:

1. **Medical Knowledge Retriever** — answers questions using condition/ICD-10 reference documents.
2. **Patient History Retriever** — embeds historical patient encounters to surface similar past cases for a new patient query.

Both retrievers are orchestrated through LangChain, combined into a single context, and passed to a chat LLM to generate a decision-support style answer with citations.

**Config used in this notebook:**
- Embedding model: `sentence-transformers/all-MiniLM-L6-v2`
- Vector DB: `Chroma`
- Chat model: `gpt-5-nano`

### 0. ⚙️ Setup & Installation

📝 Details installation and setup instruction is present in the "README.md" file

### 1. 📩 Importing package and set up reading environment  and set up models

In [ ]:
import os
import pandas as pd
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
import gradio as gr



In [ ]:
# Load environment variables from .env file
load_dotenv()
# ---- CONFIG ----
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CHAT_MODEL_NAME = "gpt-5-nano"

### 2. 🗃️ Load & Prepare Data

We merge the encounters file (clinical notes + dx codes) with the patients file (demographics) on `patient_id`.


In [ ]:
# Read the CSV files into pandas DataFrames
encounters = pd.read_csv("sample_data/healthcare_encounters.csv")
patients = pd.read_csv("sample_data/healthcare_patients.csv")

# combine the two dataframes on patient_id
df = encounters.merge(patients, on="patient_id", how="left")
print(df.shape)
df.head()


### 3. 📖 Build the Medical Knowledge Base

- This dataset uses 9 distinct ICD-10 codes. We define short reference write-ups for each — this acts as our
"medical knowledge" corpus for Retriever
- Use any AI tool to get this information


In [ ]:
ICD10_KNOWLEDGE = {
    "I10": "Essential (primary) hypertension. Chronically elevated blood pressure with no single "
           "identifiable secondary cause. Often linked to diet, weight, stress, and genetics. "
           "Management: lifestyle changes (sodium reduction, exercise, weight loss) and antihypertensive "
           "medications such as ACE inhibitors, ARBs, calcium channel blockers, or thiazide diuretics.",

    "J45": "Asthma. A chronic inflammatory airway disease causing episodic wheezing, shortness of breath, "
           "chest tightness, and cough. Triggers include allergens, exercise, and respiratory infections. "
           "Management: inhaled corticosteroids for control, short-acting beta-agonists (e.g., albuterol) "
           "for acute symptom relief, and trigger avoidance.",

    "M54": "Dorsalgia (back pain), including sciatica. Pain along the back or radiating down the leg due to "
           "nerve root irritation, most commonly from disc issues or muscular strain. Management: physical "
           "therapy, NSAIDs, activity modification, and in chronic/severe cases, imaging and specialist referral.",

    "I25": "Chronic ischemic heart disease. Long-term reduced blood flow to the heart muscle, usually from "
           "atherosclerosis, raising risk of angina and myocardial infarction. Major risk factors include "
           "smoking, hyperlipidemia, hypertension, and diabetes. Management: statins, antiplatelet therapy, "
           "lifestyle changes, and risk factor control.",

    "E78": "Disorders of lipoprotein metabolism (hyperlipidemia). Elevated cholesterol/triglycerides that "
           "increase cardiovascular risk. Often asymptomatic, found on routine labs. Management: statins, "
           "dietary changes, exercise, and monitoring lipid panels.",

    "G47": "Sleep disorders, including insomnia and sleep apnea. Disruption of normal sleep patterns leading "
           "to fatigue and daytime impairment. Management: sleep hygiene counseling, CPAP for sleep apnea, "
           "and addressing contributing conditions like obesity or anxiety.",

    "F41": "Anxiety disorders. Persistent excessive worry or panic symptoms (e.g., palpitations, restlessness) "
           "that can mimic somatic illness. Management: cognitive behavioral therapy, SSRIs/SNRIs, and "
           "lifestyle interventions such as exercise and stress reduction.",

    "K21": "Gastroesophageal reflux disease (GERD). Backflow of stomach acid into the esophagus causing "
           "heartburn, regurgitation, and sometimes atypical chest discomfort. Management: lifestyle changes "
           "(avoid late/large meals, reduce caffeine/alcohol, weight loss, head-of-bed elevation) and "
           "acid-suppressing medications (PPIs, H2 blockers).",

    "E11": "Type 2 diabetes mellitus. Chronic condition with insulin resistance and elevated blood glucose, "
           "associated with obesity and sedentary lifestyle. Risk factor for cardiovascular disease, "
           "neuropathy, and nephropathy. Management: metformin and other glucose-lowering agents, diet, "
           "exercise, and regular A1c monitoring.",
}

knowledge_docs = [
    Document(page_content=text, metadata={"dx_code": code, "source": "knowledge_base"})
    for code, text in ICD10_KNOWLEDGE.items()
]
print(f"Built {len(knowledge_docs)} knowledge documents")

### 4. 📃 Build Patient History Documents

Each encounter is turned into a single text "case summary" combining structured demographics with the
free-text clinical note. This is what gets embedded for Retriever #2 (similar historical case lookup).


In [ ]:
def build_case_text(row):
    """
        Build a text representation of a patient case from a DataFrame row."""
    smoker = "smoker" if row["smoker"] == 1 else "non-smoker"
    diabetic = "diabetic" if row["diabetic"] == 1 else "non-diabetic"
    return (
        f"Patient: {row['age']}yo {row['sex']}, {smoker}, {diabetic}, BMI {row['bmi']}. "
        f"Note: {row['note']} Diagnosis code: {row['dx_code']}."
    )

df["case_text"] = df.apply(build_case_text, axis=1)

history_docs = [
    Document(
        page_content=row["case_text"],
        metadata={
            "encounter_id": row["encounter_id"],
            "patient_id": row["patient_id"],
            "dx_code": row["dx_code"],
            "date": row["date"],
            "source": "patient_history",
        },
    )
    for _, row in df.iterrows()
]
print(f"Built {len(history_docs)} patient history documents")


### 5. 🫙 Create Embeddings & Vector Stores

We use a single embedding model (`all-MiniLM-L6-v2`) for both corpora but store them in **two separate
Chroma collections** so we can query knowledge and patient history independently or together.


In [ ]:
# Initialize the embedding model from HuggingFace
embedding_model = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)

# Create vector stores for knowledge
knowledge_vectorstore = Chroma.from_documents(
    documents=knowledge_docs,
    embedding=embedding_model,
    collection_name="medical_knowledge",
    persist_directory="./chroma_db",
)

# Create vector stores for patient history
history_vectorstore = Chroma.from_documents(
    documents=history_docs,
    embedding=embedding_model,
    collection_name="patient_history",
    persist_directory="./chroma_db",
)

# set up retrievers for knowledge and patient history
knowledge_retriever = knowledge_vectorstore.as_retriever(search_kwargs={"k": 2})
history_retriever = history_vectorstore.as_retriever(search_kwargs={"k": 3})

print("Vector stores ready.")


### 6. 🧪 Quick Retrieval Sanity Checks

Before wiring up the LLM, confirm both retrievers return sensible results on their own.


In [ ]:
print("--- Knowledge retriever test ---")
for d in knowledge_retriever.invoke("What lifestyle changes help with GERD?"):
    print(d.metadata["dx_code"], "->", d.page_content[:100], "...")

print("\n--- Patient history retriever test ---")
for d in history_retriever.invoke(
    "65 year old male smoker with fatigue and history of hyperlipidemia"
):
    print(d.metadata["encounter_id"], d.metadata["dx_code"], "->", d.page_content[:120], "...")


### 7. 🧠 Build the LangChain RAG Chain

The chain:
1. Takes a user query (patient presentation / question).
2. Retrieves top documents from **both** vector stores.
3. Merges them into a single context block, clearly labeled by source type.
4. Sends the context + query to `gpt-5-nano` with a prompt that enforces citation and a decision-support disclaimer.


In [ ]:
llm = ChatOpenAI(model=CHAT_MODEL_NAME, temperature=0)

SYSTEM_PROMPT = """
You are MedAssist, a clinical decision-support assistant. You are NOT a doctor and must never
state a definitive diagnosis. Use only the provided context (medical knowledge excerpts and similar
historical patient cases) to answer.

Formatting Rules:
- Always respond using well-structured Markdown (headings, bold, bullet points, tables where useful).
- Use relevant medical icons/emojis (e.g., 🩺 🏥 💊 📋 🔬) sparingly to visually anchor key sections —
  do not overuse them or place them in clinical findings/disclaimers where they could look unserious.

Rules:
- If the user's message is a greeting or general opening (e.g., "hi", "hello", "hey") with no
  clinical question or context provided, respond only with:

  "🩺 **Hi, I'm MedAssist.** How can I help you today?"

  Do not cite sources, add disclaimers, or attempt to answer a clinical question in this case.

- For all other queries:
  - Structure the response in Markdown with clear sections, e.g.:
    - **## Summary** — brief plain-language answer
    - **## Supporting Context** — relevant findings from provided sources
    - **## Sources** — cite dx_code and/or encounter_id used
  - If the retrieved context is weak or irrelevant to the question, say so explicitly under a
    **## Note** section and recommend clinical evaluation instead of guessing.
  - Always end clinical answers (not greetings) with a horizontal rule and:

    > ⚠️ *This is a decision-support suggestion, not a diagnosis.*
"""

PROMPT_TEMPLATE = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human",
     "Medical Knowledge Context:\n{knowledge_context}\n\n"
     "Similar Patient History Context:\n{history_context}\n\n"
     "Question:\n{question}"),
])

def format_docs(docs, empty_msg="No relevant documents found."):
    if not docs:
        return empty_msg
    return "\n\n".join(
        f"[{d.metadata.get('dx_code', d.metadata.get('source'))} | "
        f"{d.metadata.get('encounter_id', 'knowledge_base')}] {d.page_content}"
        for d in docs
    )

def retrieve_and_format(question):
    knowledge_docs_retrieved = knowledge_retriever.invoke(question)
    history_docs_retrieved = history_retriever.invoke(question)
    return {
        "knowledge_context": format_docs(knowledge_docs_retrieved),
        "history_context": format_docs(history_docs_retrieved),
        "question": question,
    }

rag_chain = (
    RunnablePassthrough()
    | (lambda q: retrieve_and_format(q))
    | PROMPT_TEMPLATE
    | llm
    | StrOutputParser()
)

print("RAG chain ready.")


### 8. 🧩 Develop user chat interface using Gradio for using the bot


In [ ]:
def rag_chatbot(user_message, chat_history=[]):
    answer = rag_chain.invoke(user_message)
    chat_history = chat_history + [(user_message, answer)]
    return answer


In [ ]:
gr.ChatInterface(rag_chatbot, chatbot=gr.Chatbot(height=600)).launch()
